In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import joblib
import warnings
import os
from itertools import combinations
from sklearn.model_selection import train_test_split, GroupShuffleSplit, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.feature_selection import mutual_info_classif

from Bio import Align
from Bio.Align import substitution_matrices
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'
np.random.seed(42)

os.makedirs("results", exist_ok=True)

"""
print("\n>>> Module 1: Data Cleaning and External Set Splitting")

# 1.1 Clean MIC
df_mic = pd.read_csv("data/dbaasp_mic_ranking.csv")
df_mic.columns = ["ID", "COMPLEXITY", "NAME", "N_TERM", "SEQUENCE", "C_TERM", "ACTIVITY"]
df_mic["MIC_uM"] = pd.to_numeric(df_mic["ACTIVITY"].astype(str).str.replace('>', '').str.replace('<', '').str.replace('=', '').str.strip(), errors='coerce')
df_mic = df_mic.dropna(subset=["MIC_uM"]).copy()

standard_aa = set("ACDEFGHIKLMNPQRSTVWY")
df_mic["length"] = df_mic["SEQUENCE"].str.len()
df_mic = df_mic[(df_mic["length"] >= 5) & (df_mic["length"] <= 50)]
df_mic = df_mic[df_mic["SEQUENCE"].apply(lambda s: set(s).issubset(standard_aa))]
df_mic = df_mic.drop_duplicates(subset="SEQUENCE").reset_index(drop=True)
df_mic["label"] = (df_mic["MIC_uM"] <= 10).astype(int)

train_val_df, external_test_df = train_test_split(df_mic, test_size=0.15, stratify=df_mic["label"], random_state=42)
train_val_df.to_csv("data/mic_train_val.csv", index=False)
external_test_df.to_csv("data/mic_external_test.csv", index=False)
print(f"MIC cleaned: {len(df_mic)}, External: {len(external_test_df)}, Train_Val: {len(train_val_df)}")

# 1.2 Clean HC50
df_hemo_train = pd.read_csv("data/cross_val_dataset.csv")
df_hemo_train.columns = ["sequence", "HC50_uM", "original_label"]
df_hemo_train["label"] = (df_hemo_train["HC50_uM"] >= 100).astype(int)
df_hemo_train["length"] = df_hemo_train["sequence"].str.len()
df_hemo_train = df_hemo_train[(df_hemo_train["length"] >= 5) & (df_hemo_train["length"] <= 50)]
df_hemo_train = df_hemo_train[df_hemo_train["sequence"].apply(lambda s: set(s).issubset(standard_aa))]
df_hemo_train = df_hemo_train.drop_duplicates(subset="sequence").reset_index(drop=True)

df_hemo_test = pd.read_csv("data/independent_dataset.csv")
df_hemo_test.columns = ["sequence", "HC50_uM", "original_label"]
df_hemo_test["label"] = (df_hemo_test["HC50_uM"] >= 100).astype(int)
df_hemo_test["length"] = df_hemo_test["sequence"].str.len()
df_hemo_test = df_hemo_test[(df_hemo_test["length"] >= 5) & (df_hemo_test["length"] <= 50)]
df_hemo_test = df_hemo_test[df_hemo_test["sequence"].apply(lambda s: set(s).issubset(standard_aa))]
df_hemo_test = df_hemo_test.drop_duplicates(subset="sequence").reset_index(drop=True)

df_hemo_train.to_csv("data/hemo_train_clean.csv", index=False)
df_hemo_test.to_csv("data/hemo_external_test.csv", index=False)
print(f"HC50 cleaned: {len(df_hemo_train)}, External: {len(df_hemo_test)}")
"""

print("\n>>> Module 2: Homology-Aware Splitting (80% Identity)")

def cluster_split(csv_in, csv_out_prefix, seq_col):
    df = pd.read_csv(csv_in)
    df["seq_id"] = df.index
    G = nx.Graph()
    G.add_nodes_from(df["seq_id"])
    seqs = df[seq_col].tolist()
    n = len(seqs)
    
    for i, j in combinations(range(n), 2):
        s1, s2 = seqs[i], seqs[j]
        if abs(len(s1) - len(s2)) > max(len(s1), len(s2)) * 0.6: continue
        matches = sum(1 for a, b in zip(s1, s2) if a == b)
        if matches / min(len(s1), len(s2)) >= 0.8:
            G.add_edge(i, j)
            
    clusters = list(nx.connected_components(G))
    cmap = {node: cid for cid, nodes in enumerate(clusters) for node in nodes}
    df["cluster"] = df["seq_id"].map(cmap)
    
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
    tr_idx, temp_idx = next(gss1.split(df, groups=df["cluster"]))
    tr_df, temp_df = df.iloc[tr_idx].copy(), df.iloc[temp_idx].copy()
    
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    val_idx, te_idx = next(gss2.split(temp_df, groups=temp_df["cluster"]))
    val_df, te_df = temp_df.iloc[val_idx].copy(), temp_df.iloc[te_idx].copy()
    
    tr_df.to_csv(f"data/{csv_out_prefix}_train_split.csv", index=False)
    val_df.to_csv(f"data/{csv_out_prefix}_val_split.csv", index=False)
    te_df.to_csv(f"data/{csv_out_prefix}_internal_test_split.csv", index=False)
    print(f"{csv_out_prefix}: Train={len(tr_df)}, Val={len(val_df)}, Test={len(te_df)}, Clusters={len(clusters)}")

cluster_split("data/mic_train_val.csv", "mic", "SEQUENCE")
cluster_split("data/hemo_train_clean.csv", "hemo", "sequence")



print("\n>>> Module 3: Data Leakage Audit (Filtering External Sets)")

def filter_homology(ext_csv, train_csv, seq_col, out_csv):
    ext_df = pd.read_csv(ext_csv)
    train_seqs = pd.read_csv(train_csv)[seq_col].tolist()
    keep_idx = []
    for i, seq_e in enumerate(ext_df[seq_col]):
        is_homolog = False
        for seq_t in train_seqs:
            if abs(len(seq_e) - len(seq_t)) > max(len(seq_e), len(seq_t)) * 0.6: continue
            if sum(1 for a, b in zip(seq_e, seq_t) if a == b) / min(len(seq_e), len(seq_t)) >= 0.8:
                is_homolog = True
                break
        if not is_homolog: keep_idx.append(i)
    
    clean_df = ext_df.iloc[keep_idx].reset_index(drop=True)
    clean_df.to_csv(out_csv, index=False)
    print(f"{out_csv}: {len(ext_df)} -> {len(clean_df)} peptides (removed {len(ext_df)-len(clean_df)})")

filter_homology("data/mic_external_test.csv", "data/mic_train_val.csv", "SEQUENCE", "data/mic_external_test_clean.csv")
filter_homology("data/hemo_external_test.csv", "data/hemo_train_clean.csv", "sequence", "data/hemo_external_test_clean.csv")


print("\n>>> Module 4: Feature Extraction")

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")

def get_22d(seqs):
    feat = []
    for s in seqs:
        L = len(s)
        aac = [s.count(aa) / L for aa in AMINO_ACIDS]
        q = s.count("K") + s.count("R") - s.count("D") - s.count("E")
        feat.append(aac + [L, q])
    return np.array(feat)

def get_dpc(seqs):
    feat = []
    for s in seqs:
        L = len(s)
        dpc = []
        for aa1 in AMINO_ACIDS:
            for aa2 in AMINO_ACIDS:
                dpc.append(sum(1 for i in range(L-1) if s[i]==aa1 and s[i+1]==aa2) / (L-1) if L>1 else 0)
        feat.append(dpc)
    return np.array(feat)

def get_allcomp(seqs):
    feat = []
    for s in seqs:
        L = len(s)
        aac = [s.count(aa)/L for aa in AMINO_ACIDS]
        dpc = []
        for aa1 in AMINO_ACIDS:
            for aa2 in AMINO_ACIDS:
                dpc.append(sum(1 for i in range(L-1) if s[i]==aa1 and s[i+1]==aa2) / (L-1) if L>1 else 0)
        q = s.count("K") + s.count("R") - s.count("D") - s.count("E")
        aromatic = sum(1 for aa in s if aa in "FWY")/L
        hydro = sum(1 for aa in s if aa in "AVLIMFWP")/L
        feat.append(aac + dpc + [L, q, aromatic, hydro])
    return np.array(feat)

datasets = [("mic_train", "SEQUENCE"), ("mic_val", "SEQUENCE"), ("mic_internal_test", "SEQUENCE"), ("mic_external_test", "SEQUENCE"),
            ("hemo_train", "sequence"), ("hemo_val", "sequence"), ("hemo_internal_test", "sequence"), ("hemo_external_test", "sequence")]

for prefix, col in datasets:
    if prefix == "mic_train": path = "data/mic_train_split.csv"
    elif prefix == "mic_val": path = "data/mic_val_split.csv"
    elif prefix == "mic_internal_test": path = "data/mic_internal_test_split.csv"
    elif prefix == "mic_external_test": path = "data/mic_external_test_clean.csv"
    elif prefix == "hemo_train": path = "data/hemo_train_split.csv"
    elif prefix == "hemo_val": path = "data/hemo_val_split.csv"
    elif prefix == "hemo_internal_test": path = "data/hemo_internal_test_split.csv"
    elif prefix == "hemo_external_test": path = "data/hemo_external_test_clean.csv"

    df = pd.read_csv(path)
    seqs = df[col].tolist()
    y = df["label"].values
    
    np.save(f"data/X_{prefix}.npy", get_22d(seqs))
    np.save(f"data/y_{prefix}.npy", y)
    np.save(f"data/X_{prefix}_dpc.npy", get_dpc(seqs))
    np.save(f"data/X_{prefix}_allcomp.npy", get_allcomp(seqs))
    print(f"{prefix}: {len(seqs)} samples, 22D/DPC/ALLCOMP extracted.")

def select_mrmr(X, y, n_features=100):
    n_total = X.shape[1]
    mi_y = mutual_info_classif(X, y, random_state=42)
    selected = [np.argmax(mi_y)]
    remaining = list(set(range(n_total)) - set(selected))
    
    n_features = min(n_features, n_total)
    
    for _ in range(n_features - 1):
        best_idx, best_score = -1, -np.inf
        for i in remaining:
            relevance = mi_y[i]
            redundancy = np.mean([abs(np.corrcoef(X[:, i], X[:, s])[0, 1]) for s in selected])
            score = relevance - redundancy
            if score > best_score:
                best_score = score
                best_idx = i
        selected.append(best_idx)
        remaining.remove(best_idx)
    return selected

for task in ["mic", "hemo"]:
    X_tr = np.load(f"data/X_{task}_train_allcomp.npy")
    y_tr = np.load(f"data/y_{task}_train.npy")
    print(f"Running mRMR for {task}...")
    top_features = select_mrmr(X_tr, y_tr, n_features=100)
    
    np.save(f"data/X_{task}_train_mrmr.npy", X_tr[:, top_features])
    np.save(f"data/X_{task}_internal_test_mrmr.npy", np.load(f"data/X_{task}_internal_test_allcomp.npy")[:, top_features])
print("mRMR features extracted.")


print("\n>>> Module 5: Model Training and Calibration")

param_dist = {'n_estimators': [100, 200, 500], 'max_features': ['sqrt', 'log2', 0.3, 0.5], 'min_samples_leaf': [1, 2, 4], 'max_depth': [None, 10, 20, 30]}

def train_and_save(task):
    X_tr = np.load(f"data/X_{task}_train.npy")
    y_tr = np.load(f"data/y_{task}_train.npy")
    
    rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
    search = RandomizedSearchCV(rf, param_dist, n_iter=100, cv=5, scoring='roc_auc', random_state=42, n_jobs=-1)
    search.fit(X_tr, y_tr)
    
    best_rf = search.best_estimator_
    cal = CalibratedClassifierCV(best_rf, method='sigmoid', cv=5)
    cal.fit(X_tr, y_tr)
    
    joblib.dump(best_rf, f"data/{task}_rf.pkl")
    joblib.dump(cal, f"data/{task}_calibrated.pkl")
    print(f"{task} best params: {search.best_params_}, CV AUROC: {search.best_score_:.4f}")

train_and_save("mic")
train_and_save("hemo")

def calc_ci(y, proba, n_boot=1000):
    rng = np.random.RandomState(42)
    aucs = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        aucs.append(roc_auc_score(y[idx], proba[idx]))
    return np.percentile(aucs, [2.5, 97.5])

def paired_bootstrap_p(y, p1, p2, n_boot=1000):
    rng = np.random.RandomState(42)
    diffs = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        diffs.append(roc_auc_score(y[idx], p1[idx]) - roc_auc_score(y[idx], p2[idx]))
    return 2 * min(np.mean(np.array(diffs)<=0), np.mean(np.array(diffs)>=0))

def holm_bonferroni(p_values, alpha=0.05):
    n = len(p_values)
    sorted_pairs = sorted(enumerate(p_values), key=lambda x: x[1])
    adjusted = [0] * n
    reject = [False] * n
    prev_adj = 0
    for rank, (orig_idx, p) in enumerate(sorted_pairs, start=1):
        adj = (n - rank + 1) * p
        adj = max(adj, prev_adj) 
        adjusted[orig_idx] = min(adj, 1.0)
        prev_adj = adjusted[orig_idx]
        reject[orig_idx] = adjusted[orig_idx] < alpha
    return adjusted, reject


print("\n>>> Module 6: Evaluations and 95% Confidence Intervals")

datasets_info = [
    ("MIC", "Internal", "data/mic_internal_test_split.csv", "data/X_mic_internal_test.npy", "mic"),
    ("MIC", "Leaky External", "data/mic_external_test.csv", "data/X_mic_external_test_leaky.npy", "mic"),
    ("MIC", "Clean External", "data/mic_external_test_clean.csv", "data/X_mic_external_test_clean.npy", "mic"),
    ("HC50", "Internal", "data/hemo_internal_test_split.csv", "data/X_hemo_internal_test.npy", "hemo"),
    ("HC50", "Leaky External", "data/hemo_external_test.csv", "data/X_hemo_external_test_leaky.npy", "hemo"),
    ("HC50", "Clean External", "data/hemo_external_test_clean.csv", "data/X_hemo_external_test_clean.npy", "hemo")
]

for task, col in [("mic", "SEQUENCE"), ("hemo", "sequence")]:
    df_leaky = pd.read_csv(f"data/{task}_external_test.csv")
    X_leaky = get_22d(df_leaky[col].tolist())
    np.save(f"data/X_{task}_external_test_leaky.npy", X_leaky)
    
    df_clean = pd.read_csv(f"data/{task}_external_test_clean.csv")
    X_clean = get_22d(df_clean[col].tolist())
    np.save(f"data/X_{task}_external_test_clean.npy", X_clean)


t1_data = []
fig2_auc_mic = []
fig2_auc_hemo = []

for task_name, set_name, csv_path, npy_path, model_task in datasets_info:
    model = joblib.load(f"data/{model_task}_calibrated.pkl")
    y = pd.read_csv(csv_path)["label"].values
    X = np.load(npy_path)
    proba = model.predict_proba(X)[:, 1]
    
    auc = roc_auc_score(y, proba)
    brier = brier_score_loss(y, proba)
    ci = calc_ci(y, proba)
    
    t1_data.append([task_name if set_name=="Internal" else "", set_name, str(len(y)), f"{auc:.4f}", f"{ci[0]:.3f} – {ci[1]:.3f}", f"{brier:.4f}"])
    
    if task_name == "MIC": fig2_auc_mic.append(auc)
    else: fig2_auc_hemo.append(auc)
    print(f"[{task_name}] {set_name:<15} (N={len(y):>3}): AUROC={auc:.4f}")


print("\n>>> Module 7: Feature Importance")
def group_perm_imp(model, X, y, n_repeats=100):
    np.random.seed(42); hydro_idx = [AMINO_ACIDS.index(aa) for aa in ['A','V','L','I','M','F','W','P']]
    cat_idx = [AMINO_ACIDS.index(aa) for aa in ['K','R','H']]
    groups = {"Hydrophobic": hydro_idx, "Cationic": cat_idx, "Length": [20], "Net_charge": [21]}
    
    baseline = roc_auc_score(y, model.predict_proba(X)[:, 1])
    imp = {}
    for g_name, idx in groups.items():
        scores = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            p_idx = np.random.permutation(len(X_perm))
            X_perm[:, idx] = X_perm[p_idx][:, idx]
            scores.append(baseline - roc_auc_score(y, model.predict_proba(X_perm)[:, 1]))
        imp[g_name] = np.mean(scores)
    return imp

X_val_mic = np.load("data/X_mic_val.npy")
X_val_hemo = np.load("data/X_hemo_val.npy")
mic_group_imp = group_perm_imp(joblib.load("data/mic_calibrated.pkl"), X_val_mic, np.load("data/y_mic_val.npy"))
hemo_group_imp = group_perm_imp(joblib.load("data/hemo_calibrated.pkl"), X_val_hemo, np.load("data/y_hemo_val.npy"))

mic_rf = joblib.load("data/mic_rf.pkl")
hemo_rf = joblib.load("data/hemo_rf.pkl")


print("\n>>> Module 8: High-Dimensional Comparisons & Holm Correction")
def holm_bonferroni(p_values, alpha=0.05):
    n = len(p_values)
    sorted_pairs = sorted(enumerate(p_values), key=lambda x: x[1])
    adjusted = [0] * n
    reject = [False] * n
    prev_adj = 0
    for rank, (orig_idx, p) in enumerate(sorted_pairs, start=1):
        adj = (n - rank + 1) * p
        adj = max(adj, prev_adj) 
        adjusted[orig_idx] = min(adj, 1.0)
        prev_adj = adjusted[orig_idx]
        reject[orig_idx] = adjusted[orig_idx] < alpha
    return adjusted, reject

def paired_bootstrap_p(y, p1, p2, n_boot=1000):
    rng = np.random.RandomState(42)
    diffs = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        diffs.append(roc_auc_score(y[idx], p1[idx]) - roc_auc_score(y[idx], p2[idx]))
    return 2 * min(np.mean(np.array(diffs)<=0), np.mean(np.array(diffs)>=0))


t2_data = []
for task in ["mic", "hemo"]:
    print(f"\n--- {task.upper()} Feature Comparisons ---")
    X_te = np.load(f"data/X_{task}_internal_test.npy")
    y_te = np.load(f"data/y_{task}_internal_test.npy")
    
    model_22 = joblib.load(f"data/{task}_calibrated.pkl")
    proba_22 = model_22.predict_proba(X_te)[:, 1]
    auc_22 = roc_auc_score(y_te, proba_22)
    
    target_name = "MIC" if task == "mic" else "HC50"
    t2_data.append([target_name, "Minimalist", "22", f"{auc_22:.4f}", "—"])
    
    p_values = []
    aucs_list = []
    feat_names_mrmr = ["dpc", "allcomp", "mrmr"]
    dims = ["400", "424", "100"]
    
    for feat_name in feat_names_mrmr:
        X_te_f = np.load(f"data/X_{task}_internal_test_{feat_name}.npy")
        X_tr_f = np.load(f"data/X_{task}_train_{feat_name}.npy")
        y_tr = np.load(f"data/y_{task}_train.npy")
        
        rf = RandomForestClassifier(n_estimators=200, max_features='sqrt', class_weight='balanced', random_state=42)
        rf.fit(X_tr_f, y_tr)
        proba_f = rf.predict_proba(X_te_f)[:, 1]
        auc_f = roc_auc_score(y_te, proba_f)
        aucs_list.append(auc_f)
        
        p_val = paired_bootstrap_p(y_te, proba_22, proba_f)
        p_values.append(p_val)
        print(f"{feat_name.upper():<8} AUROC: {auc_f:.4f}, Raw p = {p_val:.4f}")
        
    p_adj, _ = holm_bonferroni(p_values)
    for i, name in enumerate(["DPC", "ALLCOMP", "mRMR"]):
        t2_data.append(["", name, dims[i], f"{aucs_list[i]:.4f}", f"{p_adj[i]:.4f}"])


print("\n>>> Module 9: Sensitivity Analyses")
X_tr_mic = np.load("data/X_mic_train.npy")
X_te_mic = np.load("data/X_mic_internal_test.npy")
train_mic = pd.read_csv("data/mic_train_split.csv")
test_mic = pd.read_csv("data/mic_internal_test_split.csv")

X_tr_hemo = np.load("data/X_hemo_train.npy")
X_te_hemo = np.load("data/X_hemo_internal_test.npy")
train_hemo = pd.read_csv("data/hemo_train_split.csv")
test_hemo = pd.read_csv("data/hemo_internal_test_split.csv")

def calc_charge_ph(seq):
    PKA = {'D': 3.65, 'E': 4.25, 'H': 6.0, 'C': 8.18, 'Y': 10.07, 'K': 10.53, 'R': 12.48}
    c = 1.0/(1.0+10**(9.69-7.4)) - 1.0/(1.0+10**(7.4-2.34))
    for aa in seq:
        if aa in ['K','R','H']: c += 1.0/(1.0+10**(PKA[aa]-7.4))
        elif aa in ['D','E','C','Y']: c -= 1.0/(1.0+10**(7.4-PKA[aa]))
    return c


print("\n>>> Module 10: Multi-Objective Screening & Pareto")
cand_df = pd.read_csv("data/mic_external_test_clean.csv")
X_cand = np.load("data/X_mic_external_test_clean.npy")
mic_model = joblib.load("data/mic_calibrated.pkl")
hemo_model = joblib.load("data/hemo_calibrated.pkl")

cand_df["p_act"] = mic_model.predict_proba(X_cand)[:, 1]
cand_df["p_safe"] = hemo_model.predict_proba(X_cand)[:, 1]
cand_df = cand_df[cand_df["p_safe"] >= 0.5].copy()
cand_df["score"] = 0.5 * cand_df["p_act"] + 0.5 * cand_df["p_safe"]
cand_df = cand_df.sort_values("score", ascending=False).reset_index(drop=True)

def find_pareto(df):
    idx = []
    for i in range(len(df)):
        if not any((df.iloc[j]["p_act"] >= df.iloc[i]["p_act"] and df.iloc[j]["p_safe"] >= df.iloc[i]["p_safe"] and 
                   (df.iloc[j]["p_act"] > df.iloc[i]["p_act"] or df.iloc[j]["p_safe"] > df.iloc[i]["p_safe"]))
                   for j in range(len(df)) if i != j):
            idx.append(i)
    return df.iloc[idx]

pareto_df = find_pareto(cand_df)


t3_data = []
for i, (_, row) in enumerate(pareto_df.iterrows()):
    t3_data.append([str(i+1), row["SEQUENCE"], f"{row['p_act']:.3f}", f"{row['p_safe']:.3f}", f"{row['score']:.3f}"])


print("\n>>> Module 11: Biophysical Properties, Global Alignment & Weight Sensitivity")
HYDRO = {'A':0.62,'C':0.29,'D':-0.90,'E':-0.74,'F':1.19,'G':0.48,'H':-0.40,'I':1.38,'K':-1.50,'L':1.06,'M':0.64,'N':-0.78,'P':0.12,'Q':-0.85,'R':-2.53,'S':-0.18,'T':-0.05,'V':1.08,'W':0.81,'Y':0.26}
def calc_hm(seq):
    return abs(sum(HYDRO.get(aa,0) * np.exp(1j * np.deg2rad(100*i)) for i,aa in enumerate(seq))) / len(seq)

TEMPLATES = {"LL-37": "LLGDFKSKEKIGKEFKRIVQRIKDFLRNLVPRTES", "Magainin-2": "GIGKFLHSAKKFGKAFVGEIMNS", "Indolicidin": "ILPWKWPWWPWRR", "HNP-1": "ACYCRIPACIAGERRYGTCIYQGRLWAFCC"}
aligner = Align.PairwiseAligner(); aligner.mode = 'global'; aligner.substitution_matrix = substitution_matrices.load("BLOSUM62"); aligner.open_gap_score = -11; aligner.extend_gap_score = -1

def calc_global_identity(seq1, seq2):
    matches = aligner.align(seq1, seq2)[0].counts().identities
    return matches / min(len(seq1), len(seq2)) if min(len(seq1), len(seq2)) > 0 else 0.0


t4_data = []
for idx, row in pareto_df.iterrows():
    seq = row["SEQUENCE"]
    hm = calc_hm(seq)
    kr = seq.count("K") + seq.count("R")
    fwy = seq.count("F") + seq.count("W") + seq.count("Y")
    
    sims = {name: calc_global_identity(seq, tmpl) for name, tmpl in TEMPLATES.items()}
    closest_amp = max(sims, key=sims.get)
    max_identity = sims[closest_amp]
    q = seq.count("K") + seq.count("R") - seq.count("D") - seq.count("E")
        
    t4_data.append([seq, str(len(seq)), f"{q:+d}", f"{hm:.3f}", str(kr), str(fwy), f"{max_identity:.1%}", closest_amp])

print("\n Main Modules 2-11 Completed Successfully!")

In [ ]:
print("\n>>> Generating Manuscript Figures 2-4 and Tables 1-4")

# 1. Fig 2
labels = ['Internal\nTest', 'Leaky\nExternal', 'Clean\nExternal']
x = np.arange(len(labels)); width = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
rects1 = ax.bar(x - width/2, fig2_auc_mic, width, label='MIC Model', color='#4C72B0', edgecolor='black')
rects2 = ax.bar(x + width/2, fig2_auc_hemo, width, label='HC50 Model', color='#C44E52', edgecolor='black')
ax.set_ylabel('AUROC'); ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0.5, 0.9); ax.legend(frameon=False)
for rects in [rects1, rects2]:
    for rect in rects:
        ax.annotate(f'{rect.get_height():.3f}', xy=(rect.get_x() + rect.get_width()/2, rect.get_height()), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')
plt.tight_layout(); plt.savefig("results/Fig2_Leakage_Audit.png", dpi=300, bbox_inches='tight'); plt.close()

# 2. Fig 3
fig3_groups = ['Hydrophobic', 'Cationic', 'Length', 'Net_charge']
imp_mic = [mic_group_imp[g] for g in fig3_groups]
imp_hemo = [hemo_group_imp[g] for g in fig3_groups]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].barh(fig3_groups, imp_mic, color='#4C72B0', edgecolor='black'); axes[0].set_title("MIC Classifier")
axes[0].set_xlabel("ΔAUROC", fontsize=11); axes[0].axvline(0, color='black')
axes[1].barh(fig3_groups, imp_hemo, color='#C44E52', edgecolor='black'); axes[1].set_title("HC50 Classifier")
axes[1].set_xlabel("ΔAUROC", fontsize=11); axes[1].axvline(0, color='black')
plt.tight_layout() 
plt.savefig("results/Fig3_Group_Importance.png", dpi=300, bbox_inches='tight')
plt.close()

# 3. Fig 4
plt.figure(figsize=(6, 5))
plt.scatter(cand_df["p_act"], cand_df["p_safe"], c="gray", alpha=0.5, label="Candidates")
plt.scatter(pareto_df["p_act"], pareto_df["p_safe"], c="red", s=80, label="Pareto optimal")
plt.xlabel("Predicted Activity Probability"); plt.ylabel("Predicted Safety Probability")
plt.legend(loc='upper right', frameon=False)
plt.tight_layout(); plt.savefig("results/Fig4_Pareto_Front.png", dpi=300, bbox_inches='tight'); plt.close()


headers_t1 = ['Task', 'Dataset', 'N', 'AUROC', '95% CI', 'Brier Score']
headers_t2 = ['Target', 'Feature Set', 'Dimensionality', 'AUROC', 'Holm-adjusted p-value']
headers_t3 = ['Rank', 'Sequence', 'p_act', 'p_safe', 'Score (w=0.5)']
headers_t4 = ['Peptide', 'Length', 'Net Charge', 'HM', 'K+R', 'F+W+Y', 'Max Identity', 'Closest Known AMP']

df_t1 = pd.DataFrame(t1_data, columns=headers_t1)
df_t2 = pd.DataFrame(t2_data, columns=headers_t2)
df_t3 = pd.DataFrame(t3_data, columns=headers_t3)
df_t4 = pd.DataFrame(t4_data, columns=headers_t4)

print("\n--- Table 1. Predictive performance and data leakage audit ---")
print(df_t1.to_string(index=False))
print("\n--- Table 2. Comparison of feature configurations ---")
print(df_t2.to_string(index=False))
print("\n--- Table 3. Pareto-optimal candidate peptides ---")
print(df_t3.to_string(index=False))
print("\n--- Table 4. Biophysical properties of Pareto-optimal candidates ---")
print(df_t4.to_string(index=False))

df_t1.to_csv("results/Table1_Performance.csv", index=False)
df_t2.to_csv("results/Table2_Feature_Comparison.csv", index=False)
df_t3.to_csv("results/Table3_Pareto_Candidates.csv", index=False)
df_t4.to_csv("results/Table4_Biophysical_Properties.csv", index=False)

print("\n All manuscript figures and CSV tables generated!")



print("\n>>> Generating Supplementary Figures S1-S5 and Tables S1-S4, S6-S9")

train_mic_df = pd.read_csv("data/mic_train_split.csv")
train_hemo_df = pd.read_csv("data/hemo_train_split.csv")
test_mic_df = pd.read_csv("data/mic_internal_test_split.csv")
test_hemo_df = pd.read_csv("data/hemo_internal_test_split.csv")

# Fig S1
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(train_mic_df["length"], bins=20, color='#4C72B0', edgecolor='black', alpha=0.7)
axes[0].set_title("MIC Dataset Length Distribution"); axes[0].set_xlabel("Peptide Length (aa)"); axes[0].set_ylabel("Number of Peptides")
axes[1].hist(train_hemo_df["length"], bins=20, color='#C44E52', edgecolor='black', alpha=0.7)
axes[1].set_title("HC50 Dataset Length Distribution"); axes[1].set_xlabel("Peptide Length (aa)"); axes[1].set_ylabel("Number of Peptides")
plt.tight_layout(); plt.savefig("results/FigS1_Length_Dist.png", dpi=300, bbox_inches='tight'); plt.close()

# Fig S2
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(train_mic_df["MIC_uM"].clip(upper=50), bins=30, color='#4C72B0', edgecolor='black', alpha=0.7)
axes[0].axvline(10, color='red', linestyle='--', label='Cutoff (10 μM)')
axes[0].set_title("MIC Value Distribution"); axes[0].set_xlabel("MIC (μM)"); axes[0].set_ylabel("Number of Peptides"); axes[0].legend()
axes[1].hist(train_hemo_df["HC50_uM"].clip(upper=300), bins=30, color='#C44E52', edgecolor='black', alpha=0.7)
axes[1].axvline(100, color='red', linestyle='--', label='Cutoff (100 μM)')
axes[1].set_title("HC50 Value Distribution"); axes[1].set_xlabel("HC50 (μM)"); axes[1].set_ylabel("Number of Peptides"); axes[1].legend()
plt.tight_layout(); plt.savefig("results/FigS2_Value_Dist.png", dpi=300, bbox_inches='tight'); plt.close()

# Fig S3
def get_within_ids_from_splits(split_paths, seq_col):
    df = pd.concat([pd.read_csv(p) for p in split_paths])
    ids = []
    for _, group in df.groupby("cluster"):
        seqs = group[seq_col].tolist()
        if len(seqs) > 1:
            for s1, s2 in combinations(seqs, 2):
                ident = sum(1 for a,b in zip(s1,s2) if a==b) / min(len(s1), len(s2))
                ids.append(ident)
    return ids

mic_within = get_within_ids_from_splits(["data/mic_train_split.csv", "data/mic_val_split.csv", "data/mic_internal_test_split.csv"], "SEQUENCE")
hemo_within = get_within_ids_from_splits(["data/hemo_train_split.csv", "data/hemo_val_split.csv", "data/hemo_internal_test_split.csv"], "sequence")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(mic_within, bins=20, color='#4C72B0', edgecolor='black', alpha=0.7)
axes[0].set_title("MIC Within-Cluster Identity"); axes[0].set_xlabel("Pairwise Sequence Identity"); axes[0].set_ylabel("Number of Peptide Pairs")
axes[1].hist(hemo_within, bins=20, color='#C44E52', edgecolor='black', alpha=0.7)
axes[1].set_title("HC50 Within-Cluster Identity"); axes[1].set_xlabel("Pairwise Sequence Identity"); axes[1].set_ylabel("Number of Peptide Pairs")
plt.tight_layout(); plt.savefig("results/FigS3_Cluster_Identity.png", dpi=300, bbox_inches='tight'); plt.close()

# Fig S4
def get_max_ids(ext_csv, train_csv, seq_col):
    ext = pd.read_csv(ext_csv)[seq_col].tolist(); tr = pd.read_csv(train_csv)[seq_col].tolist()
    max_ids = []
    for e in ext:
        m = 0
        for t in tr:
            if abs(len(e)-len(t)) > max(len(e), len(t))*0.6: continue
            ident = sum(1 for a,b in zip(e,t) if a==b) / min(len(e), len(t))
            if ident > m: m = ident
        max_ids.append(m)
    return max_ids

mic_ids = get_max_ids("data/mic_external_test.csv", "data/mic_train_val.csv", "SEQUENCE")
hemo_ids = get_max_ids("data/hemo_external_test.csv", "data/hemo_train_clean.csv", "sequence")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(mic_ids, bins=20, color='#4C72B0', edgecolor='black', alpha=0.7); axes[0].axvline(0.8, color='red', linestyle='--')
axes[0].set_title("MIC External Set Max Identity"); axes[0].set_xlabel("Max Sequence Identity"); axes[0].set_ylabel("Number of Peptides")
axes[1].hist(hemo_ids, bins=20, color='#C44E52', edgecolor='black', alpha=0.7); axes[1].axvline(0.8, color='red', linestyle='--')
axes[1].set_title("HC50 External Set Max Identity"); axes[1].set_xlabel("Max Sequence Identity"); axes[1].set_ylabel("Number of Peptides")
plt.tight_layout(); plt.savefig("results/FigS4_Max_Identity.png", dpi=300, bbox_inches='tight'); plt.close()

# Fig S5
y_int_mic = test_mic_df["label"].values; y_int_hemo = test_hemo_df["label"].values
p_int_mic = mic_model.predict_proba(np.load("data/X_mic_internal_test.npy"))[:, 1]
p_int_hemo = hemo_model.predict_proba(np.load("data/X_hemo_internal_test.npy"))[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, y, proba, title, color in [(axes[0], y_int_mic, p_int_mic, "MIC Calibration", "#4C72B0"), (axes[1], y_int_hemo, p_int_hemo, "HC50 Calibration", "#C44E52")]:
    prob_true, prob_pred = calibration_curve(y, proba, n_bins=10)
    ax.plot(prob_pred, prob_true, marker='o', color=color, label='Model'); ax.plot([0, 1], [0, 1], "k--", label="Perfect")
    ax.set_title(title); ax.set_xlabel("Mean Predicted Probability"); ax.set_ylabel("Fraction of Positives"); ax.legend()
plt.tight_layout(); plt.savefig("results/FigS5_Reliability.png", dpi=300, bbox_inches='tight'); plt.close()



print("\nThe supplementary table data is being dynamically calculated...")

# T S1
tS1 = [['MIC Classifier', str(mic_rf.n_estimators), str(mic_rf.max_features), str(mic_rf.min_samples_leaf), str(mic_rf.max_depth)],
       ['HC50 Classifier', str(hemo_rf.n_estimators), str(hemo_rf.max_features), str(hemo_rf.min_samples_leaf), str(hemo_rf.max_depth)]]

# T S2
tS2 = []
for model_task in ["mic", "hemo"]:
    target = "MIC" if model_task == "mic" else "HC50"
    X_te = np.load(f"data/X_{model_task}_internal_test.npy"); y_te = np.load(f"data/y_{model_task}_internal_test.npy")
    proba_22 = joblib.load(f"data/{model_task}_calibrated.pkl").predict_proba(X_te)[:, 1]
    auc_22 = roc_auc_score(y_te, proba_22)
    p_vals = []; feat_names = ["dpc", "allcomp", "mrmr"]; labels = ["DPC", "ALLCOMP", "mRMR"]; aucs = []
    for f_name in feat_names:
        rf = RandomForestClassifier(n_estimators=200, max_features='sqrt', class_weight='balanced', random_state=42)
        rf.fit(np.load(f"data/X_{model_task}_train_{f_name}.npy"), np.load(f"data/y_{model_task}_train.npy"))
        p_f = rf.predict_proba(np.load(f"data/X_{model_task}_internal_test_{f_name}.npy"))[:, 1]
        aucs.append(roc_auc_score(y_te, p_f)); p_vals.append(paired_bootstrap_p(y_te, proba_22, p_f))
    p_adj, _ = holm_bonferroni(p_vals)
    for i, label in enumerate(labels):
        tS2.append([target if i==0 else "", f"22D vs {label}", f"{aucs[i]-auc_22:+.4f}", f"{p_vals[i]:.4f}", f"{p_adj[i]:.4f}", "Yes" if p_adj[i]<0.05 else "No"])

# T S3
tS3 = [[g, f"{mic_group_imp[g]:.4f}", f"{hemo_group_imp[g]:.4f}"] for g in ['Hydrophobic', 'Cationic', 'Length', 'Net_charge']]

# T S4
feat_names = AMINO_ACIDS + ["Length", "Net_Charge"]
gini_m = pd.DataFrame({"F": feat_names, "G": mic_rf.feature_importances_}).sort_values("G", ascending=False).head(5)
gini_h = pd.DataFrame({"F": feat_names, "G": hemo_rf.feature_importances_}).sort_values("G", ascending=False).head(5)
tS4 = [[str(i+1), gini_m.iloc[i]["F"], f"{gini_m.iloc[i]['G']:.4f}", gini_h.iloc[i]["F"], f"{gini_h.iloc[i]['G']:.4f}"] for i in range(5)]


# T S6
tS6 = []
for thresh in [8, 16, 32]:
    y_tr = (train_mic_df["MIC_uM"] <= thresh).astype(int); y_te = (test_mic_df["MIC_uM"] <= thresh).astype(int)
    auc = roc_auc_score(y_te, RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42).fit(np.load("data/X_mic_train.npy"), y_tr).predict_proba(np.load("data/X_mic_internal_test.npy"))[:, 1])
    tS6.append(['MIC', f'MIC ≤ {thresh} μM', f'{auc:.4f}'])

X_tr_ph = np.load("data/X_mic_train.npy").copy(); X_te_ph = np.load("data/X_mic_internal_test.npy").copy()
X_tr_ph[:, 21] = [calc_charge_ph(s) for s in train_mic_df["SEQUENCE"]]
X_te_ph[:, 21] = [calc_charge_ph(s) for s in test_mic_df["SEQUENCE"]]
auc_ph = roc_auc_score(test_mic_df["label"], RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42).fit(X_tr_ph, train_mic_df["label"]).predict_proba(X_te_ph)[:, 1])
tS6.append(['MIC', 'pH 7.4 Charge', f'{auc_ph:.4f}'])

for thresh in [50, 200]:
    y_tr = (train_hemo_df["HC50_uM"] >= thresh).astype(int); y_te = (test_hemo_df["HC50_uM"] >= thresh).astype(int)
    auc = roc_auc_score(y_te, RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42).fit(np.load("data/X_hemo_train.npy"), y_tr).predict_proba(np.load("data/X_hemo_internal_test.npy"))[:, 1])
    tS6.append(['HC50', f'HC50 ≥ {thresh} μM', f'{auc:.4f}'])

# T S7
cand_df_60 = cand_df.copy()
for w in [0.3, 0.5, 0.7]:
    cand_df_60[f"rank_{w}"] = (w * cand_df_60["p_act"] + (1-w) * cand_df_60["p_safe"]).rank(ascending=False)
tS7 = [['w = 0.3 vs w = 0.5', f'{spearmanr(cand_df_60["rank_0.3"], cand_df_60["rank_0.5"])[0]:.4f}', f'{len(set(cand_df_60.nsmallest(20, "rank_0.3").index) & set(cand_df_60.nsmallest(20, "rank_0.5").index))}/20'], 
       ['w = 0.5 vs w = 0.7', f'{spearmanr(cand_df_60["rank_0.7"], cand_df_60["rank_0.5"])[0]:.4f}', f'{len(set(cand_df_60.nsmallest(20, "rank_0.7").index) & set(cand_df_60.nsmallest(20, "rank_0.5").index))}/20']]

# T S8
tS8 = [[f'p_act ≥ {ta:.2f}'] + [str(len(cand_df_60[(cand_df_60["p_act"] >= ta) & (cand_df_60["p_safe"] >= ts)])) for ts in [0.5, 0.6, 0.7, 0.75, 0.8]] for ta in [0.5, 0.6, 0.7, 0.8]]

# T S9
def get_bin_counts(seq_list):
    bins = {"<20%": 0, "20-40%": 0, "40-60%": 0, ">60%": 0}
    for seq in seq_list:
        max_id = max(calc_global_identity(seq, tmpl) for tmpl in TEMPLATES.values())
        if max_id < 0.20: bins["<20%"] += 1
        elif max_id < 0.40: bins["20-40%"] += 1
        elif max_id < 0.60: bins["40-60%"] += 1
        else: bins[">60%"] += 1
    return bins

pool_60_bins = get_bin_counts(cand_df_60["SEQUENCE"].tolist()); pareto_bins = get_bin_counts(pareto_df["SEQUENCE"].tolist())
tS9 = [['Safety-Filtered Pool (60)', str(pool_60_bins['<20%']), str(pool_60_bins['20-40%']), str(pool_60_bins['40-60%']), str(pool_60_bins['>60%']), '60'], ['Pareto Candidates (6)', str(pareto_bins['<20%']), str(pareto_bins['20-40%']), str(pareto_bins['40-60%']), str(pareto_bins['>60%']), '6']]



headers_s1 = ['Model', 'n_estimators', 'max_features', 'min_samples_leaf', 'max_depth']
headers_s2 = ['Target', 'Comparison', 'ΔAUROC', 'Raw p-value', 'Holm-adjusted p-value', 'Significant?']
headers_s3 = ['Feature Group', 'MIC Importance (ΔAUROC)', 'HC50 Importance (ΔAUROC)']
headers_s4 = ['Rank', 'MIC Feature', 'MIC Gini', 'HC50 Feature', 'HC50 Gini']
headers_s6 = ['Model', 'Parameter Setting', 'AUROC']
headers_s7 = ['Weighting Scheme', 'Spearman ρ', 'Top-20 Overlap']
headers_s8 = ['Act \\ Saf', '≥ 0.50', '≥ 0.60', '≥ 0.70', '≥ 0.75', '≥ 0.80']
headers_s9 = ['Sequence Group', '<20% Identity', '20–40% Identity', '40–60% Identity', '>60% Identity', 'Total']

df_s1 = pd.DataFrame(tS1, columns=headers_s1)
df_s2 = pd.DataFrame(tS2, columns=headers_s2)
df_s3 = pd.DataFrame(tS3, columns=headers_s3)
df_s4 = pd.DataFrame(tS4, columns=headers_s4)
df_s6 = pd.DataFrame(tS6, columns=headers_s6)
df_s7 = pd.DataFrame(tS7, columns=headers_s7)
df_s8 = pd.DataFrame(tS8, columns=headers_s8)
df_s9 = pd.DataFrame(tS9, columns=headers_s9)

print("\n--- Table S1. Selected hyperparameters ---")
print(df_s1.to_string(index=False))
print("\n--- Table S2. Comparison of feature sets ---")
print(df_s2.to_string(index=False))
print("\n--- Table S3. Group-level feature importance ---")
print(df_s3.to_string(index=False))
print("\n--- Table S4. Top 5 single-feature Gini importance ---")
print(df_s4.to_string(index=False))
print("\n--- Table S6. Sensitivity analysis ---")
print(df_s6.to_string(index=False))
print("\n--- Table S7. Weight sensitivity ---")
print(df_s7.to_string(index=False))
print("\n--- Table S8. High-confidence candidate count ---")
print(df_s8.to_string(index=False))
print("\n--- Table S9. Max identity distribution ---")
print(df_s9.to_string(index=False))

df_s1.to_csv("results/TableS1_Hyperparameters.csv", index=False)
df_s2.to_csv("results/TableS2_Feature_Sets.csv", index=False)
df_s3.to_csv("results/TableS3_Group_Importance.csv", index=False)
df_s4.to_csv("results/TableS4_Gini_Importance.csv", index=False)
df_s6.to_csv("results/TableS6_Sensitivity_Analysis.csv", index=False)
df_s7.to_csv("results/TableS7_Weight_Sensitivity.csv", index=False)
df_s8.to_csv("results/TableS8_Candidate_Count.csv", index=False)
df_s9.to_csv("results/TableS9_Max_Identity_Dist.csv", index=False)

print("\n All supplementary figures and CSV tables generated!")